# Attention-Guided Knowledge Distillation from ResNet-50 to MobileNetV2
## Fine-Grained Bird Classification on CUB-200-2011

This notebook implements a complete **knowledge distillation (KD)** pipeline that
transfers knowledge from a large **ResNet-50** *teacher* into a lightweight
**MobileNetV2** *student* for the **CUB-200-2011** bird classification task
(all 200 classes).

We run five experiments:

| ID | Model | Training objective |
|----|-------|--------------------|
| **E1** | ResNet-50 (teacher) | CrossEntropy |
| **E2** | MobileNetV2 (student baseline) | CrossEntropy |
| **E3** | MobileNetV2 | CrossEntropy + **Response-Based / Hinton KD** |
| **E4** | MobileNetV2 | CrossEntropy + **Attention Transfer** |
| **E5** | MobileNetV2 | CrossEntropy + **Hinton KD + Attention Transfer** (proposed) |

---

### Why knowledge distillation?
A small model trained alone only sees *hard* one-hot labels. A large teacher,
however, produces *soft* probability distributions that encode **how classes
relate to each other** ("dark knowledge"). By forcing the student to mimic the
teacher's softened outputs (and, in attention transfer, *where* the teacher
looks), the student learns a richer signal than the labels alone provide — often
reaching higher accuracy than the same architecture trained from scratch.

### Why **ResNet-50** as the teacher?
ResNet-50 is a deep, high-capacity CNN with residual connections that trains
reliably and reaches strong accuracy on fine-grained tasks. Pretrained on
ImageNet, it already encodes generic visual features (edges, textures, parts),
making it an excellent source of knowledge.

### Why **MobileNetV2** as the student?
MobileNetV2 is designed for mobile/edge deployment: inverted residual blocks and
depthwise-separable convolutions make it small and fast (~3.5M params vs.
~25M for ResNet-50). It is the perfect candidate to *receive* knowledge — we want
teacher-level accuracy at a fraction of the cost.

### Why **CUB-200-2011** for fine-grained classification?
CUB-200-2011 contains 11,788 images of **200 bird species**. The classes differ
only by subtle cues (beak shape, wing pattern, plumage colour), so the task is
**fine-grained**: global shape is not enough, the model must attend to small
discriminative regions. This makes it an ideal benchmark for attention transfer,
which explicitly tells the student *where* to look.

## 2. Imports and Configuration

We import everything up front and define one global `CFG` object holding all
hyperparameters. Centralising configuration keeps the notebook easy to modify:
change a value here and every experiment picks it up.

In [ ]:
import os
import time
import copy
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

print("PyTorch version :", torch.__version__)
print("Torchvision     :", torchvision.__version__)

In [ ]:
# ----- Device selection: CUDA if available, otherwise CPU -----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ----- Global configuration -----
@dataclass
class Config:
    image_size: int = 224
    num_classes: int = 200
    batch_size: int = 32
    num_epochs: int = 50
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4

    # Knowledge-distillation hyperparameters
    temperature: float = 4.0   # T  -> softens teacher/student logits
    alpha: float = 0.5         # weight on CrossEntropy (hard-label) loss
    beta: float = 0.5          # weight on KD (soft-label) loss
    gamma: float = 100.0       # weight on attention-transfer loss

    random_seed: int = 42

    # DataLoader workers (set to 0 on Windows / debugging)
    num_workers: int = 4

    # Where to save checkpoints
    ckpt_dir: str = "checkpoints"


CFG = Config()
os.makedirs(CFG.ckpt_dir, exist_ok=True)

# ImageNet normalisation statistics (used because models are ImageNet-pretrained)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

CFG

## 3. Dataset Loading and Splitting

CUB-200-2011 ships with metadata files that define the **official** train/test
split. We respect that split strictly:

* **Official training images** → we further split them **80/20 (stratified)**
  into our *train* and *validation* sets.
* **Official test images** → kept completely separate, used **only** for the
  final evaluation of each experiment.

Why three sets?

| Set | Purpose |
|-----|---------|
| **Train** | The model learns its weights from these images. |
| **Validation** | Used for **model selection** (best checkpoint) and **hyperparameter tuning**. Never used to update weights. |
| **Test** | Touched **only once**, at the very end, to report final accuracy. Never used for tuning. |

> **⚠️ Set your dataset path below.** Point `DATA_ROOT` to the folder that
> contains `images/`, `images.txt`, `train_test_split.txt`, and
> `image_class_labels.txt`.

In [ ]:
# ====== USER MUST SET THIS PATH ======
# DATA_ROOT should contain the CUB-200-2011 metadata files and the images/ folder:
#   DATA_ROOT/images/<class>/<image>.jpg
#   DATA_ROOT/images.txt
#   DATA_ROOT/image_class_labels.txt
#   DATA_ROOT/train_test_split.txt
DATA_ROOT = "/path/to/CUB_200_2011"   # <-- EDIT ME
# =====================================

IMAGES_DIR = os.path.join(DATA_ROOT, "images")
assert os.path.isdir(DATA_ROOT), f"DATA_ROOT not found: {DATA_ROOT} (edit the path above)"


In [ ]:
def load_cub_metadata(root):
    """Read the CUB-200-2011 metadata files and return a single DataFrame.

    Columns: img_id, filepath, label (0-indexed), is_training_img (1/0).
    """
    images = pd.read_csv(os.path.join(root, "images.txt"),
                         sep=" ", names=["img_id", "filepath"])
    labels = pd.read_csv(os.path.join(root, "image_class_labels.txt"),
                         sep=" ", names=["img_id", "label"])
    split = pd.read_csv(os.path.join(root, "train_test_split.txt"),
                        sep=" ", names=["img_id", "is_training_img"])

    df = images.merge(labels, on="img_id").merge(split, on="img_id")
    # CUB labels are 1..200; convert to 0..199 for PyTorch.
    df["label"] = df["label"] - 1
    return df


meta = load_cub_metadata(DATA_ROOT)
print("Total images:", len(meta))
print("Number of classes:", meta["label"].nunique())
print("Official train images:", int((meta.is_training_img == 1).sum()))
print("Official test images :", int((meta.is_training_img == 0).sum()))
meta.head()

In [ ]:
# ----- Build the three splits -----
# 1) Official test set stays untouched.
test_df = meta[meta.is_training_img == 0].reset_index(drop=True)

# 2) Official training set -> stratified 80/20 train/val split.
official_train = meta[meta.is_training_img == 1].reset_index(drop=True)

train_df, val_df = train_test_split(
    official_train,
    test_size=0.20,
    stratify=official_train["label"],   # keep class balance in both splits
    random_state=CFG.random_seed,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
# Sanity check: no image leaks between splits.
assert set(train_df.img_id) & set(val_df.img_id) == set()
assert set(train_df.img_id) & set(test_df.img_id) == set()
assert set(val_df.img_id) & set(test_df.img_id) == set()
print("No overlap between train / val / test. ✔")

## 4. Data Preprocessing and Augmentation

* **Training transforms** include augmentation (random crop, flip, colour jitter)
  to improve generalisation.
* **Validation / test transforms** are deterministic (resize + centre crop) so
  evaluation is reproducible.

Both pipelines normalise with **ImageNet mean/std** because all models use
ImageNet-pretrained weights and expect that input distribution.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CFG.image_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CFG.image_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
class CUBDataset(Dataset):
    """Minimal CUB-200-2011 dataset backed by a metadata DataFrame."""

    def __init__(self, df, images_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row["filepath"])
        image = Image.open(img_path).convert("RGB")   # convert handles grayscale imgs
        if self.transform is not None:
            image = self.transform(image)
        label = int(row["label"])
        return image, label

In [ ]:
# ----- Datasets -----
train_ds = CUBDataset(train_df, IMAGES_DIR, transform=train_transform)
val_ds = CUBDataset(val_df, IMAGES_DIR, transform=eval_transform)
test_ds = CUBDataset(test_df, IMAGES_DIR, transform=eval_transform)

# ----- DataLoaders -----
pin = (device.type == "cuda")
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, pin_memory=pin, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=pin)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                         num_workers=CFG.num_workers, pin_memory=pin)

# Quick peek at one batch.
xb, yb = next(iter(train_loader))
print("Batch images:", xb.shape, "| labels:", yb.shape)

## 5. Utility Functions

Reusable helpers used across all experiments: seeding, accuracy metrics,
checkpoint save/load, parameter counting, a generic evaluation loop, and an
inference-time meter. Putting them here avoids repeated code later.

In [ ]:
def set_seed(seed=42):
    """Make runs reproducible across random / numpy / torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG.random_seed)

In [ ]:
def accuracy(logits, targets):
    """Plain top-1 accuracy for a batch (fraction correct)."""
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def topk_correct(logits, targets, ks=(1, 5)):
    """Return a dict {k: number_correct_in_topk} for the batch."""
    maxk = max(ks)
    _, pred = logits.topk(maxk, dim=1, largest=True, sorted=True)  # (B, maxk)
    pred = pred.t()                                                # (maxk, B)
    correct = pred.eq(targets.view(1, -1).expand_as(pred))         # (maxk, B)
    return {k: correct[:k].reshape(-1).float().sum().item() for k in ks}

In [ ]:
def count_trainable_parameters(model):
    """Number of parameters that require gradients."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def model_size_mb(model):
    """Approximate model size in MB (params + buffers)."""
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_bytes + buffer_bytes) / (1024 ** 2)


def save_checkpoint(model, path, extra=None):
    """Save model weights (+ optional metadata) to disk."""
    payload = {"model_state": model.state_dict()}
    if extra:
        payload.update(extra)
    torch.save(payload, path)


def load_checkpoint(model, path, map_location=None):
    """Load weights into `model` from a checkpoint file. Returns the payload."""
    payload = torch.load(path, map_location=map_location or device)
    model.load_state_dict(payload["model_state"])
    return payload

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion=None):
    """Evaluate a classification model on a loader.

    Returns dict with top1, top5 accuracy and (optional) average loss.
    """
    model.eval()
    total = 0
    correct = {1: 0.0, 5: 0.0}
    loss_sum = 0.0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        if criterion is not None:
            loss_sum += criterion(logits, targets).item() * images.size(0)
        c = topk_correct(logits, targets, ks=(1, 5))
        correct[1] += c[1]
        correct[5] += c[5]
        total += images.size(0)
    out = {"top1": correct[1] / total, "top5": correct[5] / total}
    if criterion is not None:
        out["loss"] = loss_sum / total
    return out

In [ ]:
@torch.no_grad()
def measure_inference_time(model, n_warmup=5, n_runs=30, input_size=224):
    """Measure average single-image forward time in milliseconds."""
    model.eval()
    dummy = torch.randn(1, 3, input_size, input_size, device=device)
    for _ in range(n_warmup):           # warm up (CUDA kernels / caches)
        _ = model(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(n_runs):
        _ = model(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - start) / n_runs * 1000.0  # ms per image

### A generic training loop

Every experiment trains a model with the same skeleton (loop over epochs,
compute a loss, validate, keep the best checkpoint). The only thing that changes
is **how the per-batch loss is computed**. We capture that variability with a
`loss_fn(model, images, targets)` callback, so each experiment supplies just its
loss and reuses this one robust loop.

In [ ]:
def train_model(model, loss_fn, train_loader, val_loader, ckpt_path,
                num_epochs=None, lr=None, weight_decay=None, tag="model"):
    """Train `model`, select the best epoch on validation top-1, save it.

    Parameters
    ----------
    loss_fn : callable(model, images, targets) -> (loss, logits)
        Computes the training loss for one batch and returns the student logits
        (used for live train-accuracy logging).
    Returns
    -------
    history : dict of lists for plotting (train_loss, val_loss, val_top1).
    """
    num_epochs = num_epochs or CFG.num_epochs
    lr = lr or CFG.learning_rate
    weight_decay = weight_decay if weight_decay is not None else CFG.weight_decay

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {"train_loss": [], "val_loss": [], "val_top1": [], "val_top5": []}
    best_val = -1.0

    for epoch in range(num_epochs):
        model.train()
        running_loss, n = 0.0, 0
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            loss, _ = loss_fn(model, images, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            n += images.size(0)
        scheduler.step()

        train_loss = running_loss / n
        val = evaluate(model, val_loader, criterion=nn.CrossEntropyLoss())

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val["loss"])
        history["val_top1"].append(val["top1"])
        history["val_top5"].append(val["top5"])

        # Keep the best checkpoint by validation top-1 (model selection).
        if val["top1"] > best_val:
            best_val = val["top1"]
            save_checkpoint(model, ckpt_path,
                            extra={"epoch": epoch, "val_top1": best_val})
            star = "  <-- best"
        else:
            star = ""

        print(f"[{tag}] epoch {epoch+1:02d}/{num_epochs} | "
              f"train_loss {train_loss:.4f} | val_loss {val['loss']:.4f} | "
              f"val_top1 {val['top1']*100:.2f}% | val_top5 {val['top5']*100:.2f}%{star}")

    print(f"[{tag}] best val top1: {best_val*100:.2f}%  (saved to {ckpt_path})")
    return history

### Model builder functions

Small factory functions that load the ImageNet-pretrained backbones and swap the
final classifier for 200 outputs. Activations are left **unchanged**: ResNet-50
keeps **ReLU**, MobileNetV2 keeps **ReLU6**.

In [ ]:
def build_resnet50(num_classes=CFG.num_classes, pretrained=True):
    """ResNet-50 (ImageNet) with a fresh 200-way classifier head."""
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)   # replace 1000-way head
    return model.to(device)


def build_mobilenetv2(num_classes=CFG.num_classes, pretrained=True):
    """MobileNetV2 (ImageNet) with a fresh 200-way classifier head."""
    weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.mobilenet_v2(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(device)

## 6. Teacher Model — ResNet-50 (Experiment E1)

We train the teacher with plain **CrossEntropyLoss**, **AdamW** optimizer and a
**CosineAnnealingLR** schedule. The best checkpoint (by validation top-1) is
saved and then evaluated on the test set.

After training, the teacher is **frozen** (`requires_grad=False`, `eval()` mode)
and reused as a fixed knowledge source in E3–E5 — it is never updated again.

In [ ]:
TEACHER_CKPT = os.path.join(CFG.ckpt_dir, "e1_resnet50_teacher.pt")

def ce_loss_fn(model, images, targets):
    """Standard supervised loss; returns (loss, logits)."""
    logits = model(images)
    loss = F.cross_entropy(logits, targets)
    return loss, logits

In [ ]:
# ---- Train the teacher (E1) ----
set_seed(CFG.random_seed)
teacher = build_resnet50(pretrained=True)
print("Teacher trainable params:", f"{count_trainable_parameters(teacher):,}")

hist_e1 = train_model(
    teacher, ce_loss_fn, train_loader, val_loader,
    ckpt_path=TEACHER_CKPT, tag="E1-ResNet50",
)

In [ ]:
# ---- Evaluate best teacher on the TEST set ----
load_checkpoint(teacher, TEACHER_CKPT)
res_e1 = evaluate(teacher, test_loader)
print(f"E1 ResNet-50 teacher | test top1 {res_e1['top1']*100:.2f}% | "
      f"test top5 {res_e1['top5']*100:.2f}%")

In [ ]:
def freeze_teacher(model):
    """Put the teacher into eval mode and stop all gradients."""
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model

# Load best teacher weights and freeze for KD experiments.
teacher = build_resnet50(pretrained=False)
load_checkpoint(teacher, TEACHER_CKPT)
freeze_teacher(teacher)
print("Teacher loaded and frozen. Trainable params now:",
      count_trainable_parameters(teacher))

## 7. Baseline Student — MobileNetV2 (Experiment E2)

We train MobileNetV2 **without any distillation**, using only CrossEntropyLoss.
This is the reference point: every KD experiment (E3–E5) must beat this number to
demonstrate that distillation actually helps.

In [ ]:
BASELINE_CKPT = os.path.join(CFG.ckpt_dir, "e2_mobilenetv2_baseline.pt")

set_seed(CFG.random_seed)
student_baseline = build_mobilenetv2(pretrained=True)
print("Student trainable params:", f"{count_trainable_parameters(student_baseline):,}")

hist_e2 = train_model(
    student_baseline, ce_loss_fn, train_loader, val_loader,
    ckpt_path=BASELINE_CKPT, tag="E2-MobileNetV2",
)

In [ ]:
load_checkpoint(student_baseline, BASELINE_CKPT)
res_e2 = evaluate(student_baseline, test_loader)
print(f"E2 MobileNetV2 baseline | test top1 {res_e2['top1']*100:.2f}% | "
      f"test top5 {res_e2['top5']*100:.2f}%")

## 8. Knowledge Distillation Loss Functions

We now define the building blocks shared by E3–E5.

### 8.1 Response-Based / Hinton KD
The student is trained to match the teacher's **softened class distribution**:

$$\mathcal{L}_{KD} = T^2 \cdot \mathrm{KL}\big(\;\sigma(z_s/T)\;\|\;\sigma(z_t/T)\;\big)$$

* **Temperature $T$** softens the logits. A higher $T$ reveals more of the
  teacher's "dark knowledge" — the relative probabilities it assigns to *wrong*
  classes (e.g. that a sparrow looks a bit like a finch). The $T^2$ factor keeps
  the gradient magnitude comparable to the hard-label loss.
* **$\alpha$ / $\beta$** balance the two objectives in the total loss:
  $\alpha$ weights the **hard-label** CrossEntropy (ground truth) and $\beta$
  weights the **soft-label** KD term (teacher mimicry).

In [ ]:
def hinton_kd_loss(student_logits, teacher_logits, T=CFG.temperature):
    """Response-based KD loss (Hinton et al., 2015).

    KL divergence between softened teacher and student distributions, scaled by
    T^2 so its gradient is on the same scale as the hard-label loss.
    """
    student_log_prob = F.log_softmax(student_logits / T, dim=1)
    teacher_prob = F.softmax(teacher_logits / T, dim=1)
    # batchmean = average KL over the batch (correct probabilistic scaling).
    kd = F.kl_div(student_log_prob, teacher_prob, reduction="batchmean") * (T * T)
    return kd

### 8.2 Attention Transfer (AT) Loss
Instead of matching outputs, AT matches **where each network focuses**. From a
feature map $F \in \mathbb{R}^{C \times H \times W}$ we build a single-channel
**spatial attention map** by summing squared activations across channels:

$$A = \frac{1}{C}\sum_{c=1}^{C} F_c^2 \quad\Rightarrow\quad A \in \mathbb{R}^{H \times W}$$

Each attention map is **flattened and L2-normalised** so we compare *shape of
attention*, not magnitude. We then take **MSE** between teacher and student
attention vectors. If the spatial sizes differ, the student map is resized to
the teacher's size with **bilinear interpolation**.

This transfers the teacher's *focus* (e.g. "look at the beak and wing") without
forcing the student to copy exact feature values — important since the two
architectures have very different feature spaces.

In [ ]:
def attention_map(feat):
    """Channel-wise spatial attention: mean of squared activations -> (B, H, W)."""
    return feat.pow(2).mean(dim=1)


def at_loss(student_feat, teacher_feat):
    """Attention-transfer loss between two feature maps (Zagoruyko & Komodakis)."""
    s_att = attention_map(student_feat)   # (B, Hs, Ws)
    t_att = attention_map(teacher_feat)   # (B, Ht, Wt)

    # Resize student attention to teacher spatial size if needed (bilinear).
    if s_att.shape[-2:] != t_att.shape[-2:]:
        s_att = F.interpolate(s_att.unsqueeze(1), size=t_att.shape[-2:],
                              mode="bilinear", align_corners=False).squeeze(1)

    # Flatten and L2-normalise each sample's attention vector.
    s_flat = F.normalize(s_att.flatten(1), p=2, dim=1)
    t_flat = F.normalize(t_att.flatten(1), p=2, dim=1)
    return F.mse_loss(s_flat, t_flat)

### 8.3 Combined KD Loss
The proposed objective (E5) combines all three terms:

$$\mathcal{L}_{total} = \alpha\,\mathcal{L}_{CE} + \beta\,\mathcal{L}_{KD} + \gamma\,\mathcal{L}_{AT}$$

with defaults $\alpha=0.5,\ \beta=0.5,\ \gamma=100,\ T=4$. $\gamma$ is large
because the AT loss (an MSE between normalised vectors) is numerically tiny and
needs up-weighting to influence training.

In [ ]:
def combined_kd_loss(
    student_logits,
    teacher_logits,
    student_feat,
    teacher_feat,
    targets,
    use_kd=False,
    use_at=False,
    temperature=4.0,
    alpha=0.7,
    beta=0.3,
    gamma=100.0,
):
    """General KD objective. Toggle use_kd / use_at to recover E3, E4 or E5.

    All KD hyperparameters are explicit arguments (no hidden dependence on the
    global CFG), so each experiment can pass its own values. Behavior:
        use_kd & not use_at -> alpha*CE + beta*KD
        not use_kd & use_at -> CE + gamma*AT
        use_kd & use_at     -> alpha*CE + beta*KD + gamma*AT
        neither             -> CE
    """
    ce = F.cross_entropy(student_logits, targets)
    parts = {"ce": ce.item()}

    # Hard-label term is weighted by alpha only when KD is active; otherwise
    # the plain CrossEntropy is used as the base loss.
    if use_kd:
        kd = hinton_kd_loss(student_logits, teacher_logits, T=temperature)
        total = alpha * ce + beta * kd
        parts["kd"] = kd.item()
    else:
        total = ce

    if use_at:
        at = at_loss(student_feat, teacher_feat)
        total = total + gamma * at
        parts["at"] = at.item()
    return total, parts

## 9. Feature Extraction Hooks

For attention transfer we need intermediate feature maps:

* **ResNet-50** → output of **`layer4`** (last residual stage, 7×7×2048).
* **MobileNetV2** → output of its **final convolution block** (`features[-1]`,
  7×7×1280).

We use a tiny wrapper that registers a **forward hook** on the chosen module and
stores its output. This is clean, readable, and leaves the original models
untouched. The teacher wrapper is used in eval/frozen mode.

In [ ]:
class FeatureExtractor:
    """Wrap a model and capture one intermediate layer's output via a hook.

    Usage:
        fe = FeatureExtractor(model, layer)
        logits = fe(x)          # forward pass
        feat = fe.features       # captured feature map (B, C, H, W)
    """
    def __init__(self, model, layer):
        self.model = model
        self.features = None
        self._handle = layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self.features = out

    def __call__(self, x):
        return self.model(x)

    def remove(self):
        self._handle.remove()


# Layer references for each architecture.
def get_resnet_feature_layer(model):
    return model.layer4            # last residual stage

def get_mobilenet_feature_layer(model):
    return model.features[-1]      # final conv feature block

### Method-specific KD configurations

E3, E4 and E5 each get their own hyperparameters (learning rate, weight decay,
temperature, KD weights, attention weight and early-stopping patience) instead of
sharing the global defaults. Keeping them in one `KD_CONFIGS` dictionary makes
each experiment's setup explicit and easy to tune independently.

In [ ]:
KD_CONFIGS = {
    "E3-HintonKD": {
        "num_epochs": 60,
        "lr": 5e-5,
        "weight_decay": 5e-4,
        "temperature": 4.0,
        "alpha": 0.5,
        "beta": 0.5,
        "gamma": CFG.gamma,   # unused for E3 (use_at=False)
        "early_stopping_patience": 10,
    },

    "E4-AttentionTransfer": {
        "num_epochs": 60,
        "lr": 5e-5,
        "weight_decay": 5e-4,
        "temperature": CFG.temperature,  # unused for E4 (use_kd=False)
        "alpha": CFG.alpha,              # unused for E4
        "beta": CFG.beta,                # unused for E4
        "gamma": 50.0,
        "early_stopping_patience": 10,
    },

    "E5-KD+AT": {
        "num_epochs": 60,
        "lr": 5e-5,
        "weight_decay": 5e-4,
        "temperature": 4.0,
        "alpha": 0.5,
        "beta": 0.5,
        "gamma": 50.0,
        "early_stopping_patience": 10,
    },
}

These configs let E3, E4 and E5 use **individual** learning rates, weight decay,
KD weights, temperature and attention-loss weights. **Early stopping** is added to
avoid unnecessary training once validation Top-1 stops improving. **Validation
Top-1** is used for both checkpoint selection and early stopping because
classification accuracy is the main metric for this project (not validation
loss).

### A reusable KD training loop

E3, E4 and E5 differ only in which loss terms are active. We write **one** KD
training function with `use_kd` / `use_at` flags so each experiment is a
one-line call and can be run independently.

In [ ]:
def train_kd(
    teacher,
    ckpt_path,
    tag,
    use_kd,
    use_at,
    num_epochs=None,
    lr=None,
    weight_decay=None,
    temperature=None,
    alpha=None,
    beta=None,
    gamma=None,
    early_stopping_patience=None,
):
    """Train a fresh MobileNetV2 student with the frozen teacher.

    Method-specific hyperparameters override the global CFG defaults when given.
    Checkpoint selection AND early stopping are both driven by validation Top-1
    accuracy (not validation loss).

    use_kd / use_at select the experiment:
        E3 -> use_kd=True,  use_at=False
        E4 -> use_kd=False, use_at=True
        E5 -> use_kd=True,  use_at=True
    """
    # ---- resolve hyperparameters (fall back to CFG when not provided) ----
    num_epochs = num_epochs or CFG.num_epochs
    lr = lr or CFG.learning_rate
    weight_decay = weight_decay if weight_decay is not None else CFG.weight_decay
    temperature = temperature if temperature is not None else CFG.temperature
    alpha = alpha if alpha is not None else CFG.alpha
    beta = beta if beta is not None else CFG.beta
    gamma = gamma if gamma is not None else CFG.gamma

    print("=" * 80)
    print(f"Training: {tag}")
    print(f"use_kd={use_kd}, use_at={use_at}")
    print(f"epochs={num_epochs}, lr={lr}, weight_decay={weight_decay}")
    print(f"T={temperature}, alpha={alpha}, beta={beta}, gamma={gamma}")
    print(f"early_stopping_patience={early_stopping_patience}")
    print("=" * 80)

    set_seed(CFG.random_seed)
    student = build_mobilenetv2(pretrained=True)

    # Hooks for attention transfer (teacher hook only needed if use_at).
    s_fe = FeatureExtractor(student, get_mobilenet_feature_layer(student))
    t_fe = FeatureExtractor(teacher, get_resnet_feature_layer(teacher))

    freeze_teacher(teacher)  # ensure teacher is frozen + in eval mode

    optimizer = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {"train_loss": [], "val_loss": [], "val_top1": [], "val_top5": []}
    best_val = -1.0
    epochs_without_improvement = 0

    try:
        for epoch in range(num_epochs):
            student.train()                       # student trains; teacher stays eval
            current_lr = optimizer.param_groups[0]["lr"]
            running_loss, n = 0.0, 0
            for images, targets in train_loader:
                images, targets = images.to(device), targets.to(device)

                with torch.no_grad():             # teacher is frozen
                    teacher_logits = t_fe(images)
                    teacher_feat = t_fe.features

                optimizer.zero_grad()
                student_logits = s_fe(images)
                student_feat = s_fe.features

                loss, _ = combined_kd_loss(
                    student_logits, teacher_logits, student_feat, teacher_feat,
                    targets, use_kd=use_kd, use_at=use_at,
                    temperature=temperature, alpha=alpha, beta=beta, gamma=gamma,
                )
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * images.size(0)
                n += images.size(0)
            scheduler.step()

            train_loss = running_loss / n
            val = evaluate(student, val_loader, criterion=nn.CrossEntropyLoss())
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val["loss"])
            history["val_top1"].append(val["top1"])
            history["val_top5"].append(val["top5"])

            # ---- model selection + early stopping on validation Top-1 ----
            if val["top1"] > best_val:
                best_val = val["top1"]
                epochs_without_improvement = 0
                save_checkpoint(student, ckpt_path, extra={
                    "epoch": epoch,
                    "val_top1": best_val,
                    "hyperparams": {
                        "tag": tag, "use_kd": use_kd, "use_at": use_at,
                        "num_epochs": num_epochs, "lr": lr,
                        "weight_decay": weight_decay, "temperature": temperature,
                        "alpha": alpha, "beta": beta, "gamma": gamma,
                        "early_stopping_patience": early_stopping_patience,
                    },
                })
                star = "  <-- best"
            else:
                epochs_without_improvement += 1
                star = ""

            print(f"[{tag}] epoch {epoch+1:02d}/{num_epochs} | lr {current_lr:.2e} | "
                  f"train_loss {train_loss:.4f} | val_top1 {val['top1']*100:.2f}% | "
                  f"val_top5 {val['top5']*100:.2f}% | "
                  f"no_improve {epochs_without_improvement}/{early_stopping_patience}{star}")

            # Stop early if Top-1 has not improved for `patience` epochs.
            if (early_stopping_patience is not None
                    and epochs_without_improvement >= early_stopping_patience):
                print(f"[{tag}] early stopping at epoch {epoch+1} "
                      f"(no val_top1 improvement for {early_stopping_patience} epochs).")
                break
    finally:
        s_fe.remove(); t_fe.remove()              # always clean up hooks

    print(f"[{tag}] best val top1: {best_val*100:.2f}%  (saved to {ckpt_path})")
    return history

## 10. Experiment E3 — MobileNetV2 + Hinton KD

Student trained with **CrossEntropy + Response-Based KD**. The frozen ResNet-50
teacher provides softened logits; attention transfer is **off** here.

In [ ]:
cfg = KD_CONFIGS["E3-HintonKD"]
E3_CKPT = os.path.join(CFG.ckpt_dir, "e3_mobilenetv2_hintonkd.pt")

hist_e3 = history_e3 = train_kd(
    teacher=teacher,
    ckpt_path=E3_CKPT,
    tag="E3-HintonKD",
    use_kd=True,
    use_at=False,
    **cfg,
)

In [ ]:
student_e3 = build_mobilenetv2(pretrained=False)
load_checkpoint(student_e3, E3_CKPT)
res_e3 = evaluate(student_e3, test_loader)
print(f"E3 MobileNetV2 + Hinton KD | test top1 {res_e3['top1']*100:.2f}% | "
      f"test top5 {res_e3['top5']*100:.2f}%")

## 11. Experiment E4 — MobileNetV2 + Attention Transfer

Student trained with **CrossEntropy + Attention Transfer**. The student is guided
to focus on the same spatial regions as the teacher; Hinton KD is **off**.

In [ ]:
cfg = KD_CONFIGS["E4-AttentionTransfer"]
E4_CKPT = os.path.join(CFG.ckpt_dir, "e4_mobilenetv2_at.pt")

hist_e4 = history_e4 = train_kd(
    teacher=teacher,
    ckpt_path=E4_CKPT,
    tag="E4-AttentionTransfer",
    use_kd=False,
    use_at=True,
    **cfg,
)

In [ ]:
student_e4 = build_mobilenetv2(pretrained=False)
load_checkpoint(student_e4, E4_CKPT)
res_e4 = evaluate(student_e4, test_loader)
print(f"E4 MobileNetV2 + Attention Transfer | test top1 {res_e4['top1']*100:.2f}% | "
      f"test top5 {res_e4['top5']*100:.2f}%")

## 12. Experiment E5 — MobileNetV2 + Hinton KD + Attention Transfer

The **proposed method**: both KD terms active
($\mathcal{L}=\alpha\mathcal{L}_{CE}+\beta\mathcal{L}_{KD}+\gamma\mathcal{L}_{AT}$).
We expect this to perform best by combining *what* the teacher predicts with
*where* it looks.

In [ ]:
cfg = KD_CONFIGS["E5-KD+AT"]
E5_CKPT = os.path.join(CFG.ckpt_dir, "e5_mobilenetv2_kd_at.pt")

hist_e5 = history_e5 = train_kd(
    teacher=teacher,
    ckpt_path=E5_CKPT,
    tag="E5-KD+AT",
    use_kd=True,
    use_at=True,
    **cfg,
)

In [ ]:
student_e5 = build_mobilenetv2(pretrained=False)
load_checkpoint(student_e5, E5_CKPT)
res_e5 = evaluate(student_e5, test_loader)
print(f"E5 MobileNetV2 + KD + AT | test top1 {res_e5['top1']*100:.2f}% | "
      f"test top5 {res_e5['top5']*100:.2f}%")

## 13. Results Comparison

We collect every experiment's test metrics, parameter count, model size and
inference time into one table for easy comparison.

In [ ]:
def measure_for_table(builder, ckpt):
    """Load a checkpoint and return (#params, size_MB, inference_ms)."""
    m = builder(pretrained=False)
    load_checkpoint(m, ckpt)
    return (count_trainable_parameters(m), model_size_mb(m),
            measure_inference_time(m))

p_t, s_t, t_t = measure_for_table(build_resnet50, TEACHER_CKPT)
p_b, s_b, t_b = measure_for_table(build_mobilenetv2, BASELINE_CKPT)
p_3, s_3, t_3 = measure_for_table(build_mobilenetv2, E3_CKPT)
p_4, s_4, t_4 = measure_for_table(build_mobilenetv2, E4_CKPT)
p_5, s_5, t_5 = measure_for_table(build_mobilenetv2, E5_CKPT)

In [ ]:
rows = [
    ["E1", "ResNet-50",   "Teacher (CE)",       res_e1, p_t, s_t, t_t],
    ["E2", "MobileNetV2", "Baseline (CE)",      res_e2, p_b, s_b, t_b],
    ["E3", "MobileNetV2", "Hinton KD",          res_e3, p_3, s_3, t_3],
    ["E4", "MobileNetV2", "Attention Transfer", res_e4, p_4, s_4, t_4],
    ["E5", "MobileNetV2", "Hinton KD + AT",     res_e5, p_5, s_5, t_5],
]

results = pd.DataFrame([{
    "Experiment": r[0],
    "Model": r[1],
    "KD method": r[2],
    "Top-1 (%)": round(r[3]["top1"] * 100, 2),
    "Top-5 (%)": round(r[3]["top5"] * 100, 2),
    "Params (M)": round(r[4] / 1e6, 2),
    "Size (MB)": round(r[5], 2),
    "Inference (ms)": round(r[6], 2),
} for r in rows])

results

## 14. Visualization

We plot:
1. Training & validation **loss** curves per experiment.
2. Validation **top-1 accuracy** curves.
3. A final **bar chart** comparing test top-1 accuracy.
4. (Optional) Attention-map overlays for teacher vs. student.

In [ ]:
histories = {
    "E1 ResNet50": hist_e1,
    "E2 Baseline": hist_e2,
    "E3 HintonKD": hist_e3,
    "E4 AT": hist_e4,
    "E5 KD+AT": hist_e5,
}

# ----- Loss curves -----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in histories.items():
    axes[0].plot(h["train_loss"], label=name)
    axes[1].plot(h["val_loss"], label=name)
axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[1].set_title("Validation loss"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("loss")
for ax in axes: ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ----- Validation top-1 accuracy curves -----
plt.figure(figsize=(8, 5))
for name, h in histories.items():
    plt.plot([v * 100 for v in h["val_top1"]], label=name)
plt.title("Validation Top-1 Accuracy"); plt.xlabel("epoch"); plt.ylabel("top-1 (%)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ----- Final comparison bar chart (test top-1) -----
plt.figure(figsize=(8, 5))
labels = results["Experiment"] + "\n" + results["KD method"]
bars = plt.bar(labels, results["Top-1 (%)"], color="steelblue")
for b, v in zip(bars, results["Top-1 (%)"]):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f}",
             ha="center", va="bottom", fontsize=9)
plt.title("Test Top-1 Accuracy by Experiment"); plt.ylabel("top-1 (%)")
plt.xticks(rotation=15); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

### (Optional) Attention-map visualization

Overlay the spatial attention of teacher and student on sample bird images to
*see* whether the student learned to focus where the teacher does.

In [ ]:
@torch.no_grad()
def show_attention(student, teacher, dataset, n=4):
    """Visualise teacher vs. student attention maps for n random images."""
    s_fe = FeatureExtractor(student, get_mobilenet_feature_layer(student))
    t_fe = FeatureExtractor(teacher, get_resnet_feature_layer(teacher))
    student.eval(); teacher.eval()

    idxs = random.sample(range(len(dataset)), n)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    for r, i in enumerate(idxs):
        img, _ = dataset[i]
        x = img.unsqueeze(0).to(device)
        _ = t_fe(x); t_att = attention_map(t_fe.features)[0].cpu()
        _ = s_fe(x); s_att = attention_map(s_fe.features)[0].cpu()

        disp = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[r, 0].imshow(disp); axes[r, 0].set_title("input"); axes[r, 0].axis("off")
        axes[r, 1].imshow(disp)
        axes[r, 1].imshow(F.interpolate(t_att[None, None], size=disp.shape[:2],
                          mode="bilinear", align_corners=False)[0, 0],
                          cmap="jet", alpha=0.5)
        axes[r, 1].set_title("teacher attn"); axes[r, 1].axis("off")
        axes[r, 2].imshow(disp)
        axes[r, 2].imshow(F.interpolate(s_att[None, None], size=disp.shape[:2],
                          mode="bilinear", align_corners=False)[0, 0],
                          cmap="jet", alpha=0.5)
        axes[r, 2].set_title("student attn"); axes[r, 2].axis("off")

    s_fe.remove(); t_fe.remove()
    plt.tight_layout(); plt.show()

# Example (uses the E5 student + teacher):
# show_attention(student_e5, teacher, test_ds, n=4)

## 14.5 Single-Image Validation: Teacher vs. Student

A focused, intuitive sanity check: pick **one random image from the validation
set** and run **both** the teacher and the student on **that exact same image**.
For each model we show:

* the **predicted class** and whether it is **correct** (✔/✗),
* the model's **confidence** (softmax probability of its prediction),
* the **per-image CrossEntropy loss** for the true label,
* the **top-3** candidate classes with their probabilities.

This makes the teacher→student knowledge transfer concrete: you can literally see
whether the student agrees with the teacher on a single bird, how confident each
is, and how much loss each incurs on the very same input.

In [ ]:
def load_class_names(root):
    """Map 0-indexed label -> human-readable bird name from classes.txt (optional)."""
    path = os.path.join(root, "classes.txt")
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, sep=" ", names=["class_id", "name"])
    # e.g. "001.Black_footed_Albatross" -> "Black footed Albatross"
    return {int(cid) - 1: nm.split(".", 1)[-1].replace("_", " ")
            for cid, nm in zip(df.class_id, df.name)}


CLASS_NAMES = load_class_names(DATA_ROOT)
print("Loaded class names:" , "yes" if CLASS_NAMES else "no (falling back to indices)")

In [ ]:
@torch.no_grad()
def compare_on_single_image(student, teacher, dataset, idx=None, topk=3):
    """Run teacher and student on ONE image and compare prediction / loss / confidence.

    If idx is None a random image is chosen. Returns the index used (so you can
    re-run the same image). Both models see the *identical* preprocessed input.
    """
    student.eval(); teacher.eval()
    if idx is None:
        idx = random.randrange(len(dataset))

    image, true_label = dataset[idx]           # image is already transformed
    x = image.unsqueeze(0).to(device)
    target = torch.tensor([true_label], device=device)

    def analyze(model):
        logits = model(x)
        prob = F.softmax(logits, dim=1)
        loss = F.cross_entropy(logits, target).item()   # per-image CE loss
        conf, pred = prob.max(dim=1)
        topc, topi = prob.topk(topk, dim=1)
        return {
            "pred": int(pred), "conf": float(conf), "loss": loss,
            "correct": int(pred) == true_label,
            "topk": [(int(topi[0, j]), float(topc[0, j])) for j in range(topk)],
        }

    t, s = analyze(teacher), analyze(student)

    def name(lbl):
        return CLASS_NAMES[lbl] if CLASS_NAMES else f"class {lbl}"

    # ---- de-normalise the image for display ----
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    disp = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(1, 2, figsize=(12, 6),
                           gridspec_kw={"width_ratios": [1, 1.1]})
    ax[0].imshow(disp); ax[0].axis("off")
    ax[0].set_title(f"Validation image #{idx}\nTrue: {name(true_label)}")

    ax[1].axis("off")
    lines = [f"TRUE LABEL : {name(true_label)}", ""]
    for header, r in [("TEACHER  (ResNet-50)", t), ("STUDENT  (MobileNetV2)", s)]:
        mark = "OK" if r["correct"] else "WRONG"
        lines.append(header)
        lines.append(f"  pred  : {name(r['pred'])}  [{mark}]")
        lines.append(f"  conf  : {r['conf']*100:6.2f}%")
        lines.append(f"  loss  : {r['loss']:.4f}")
        lines.append(f"  top{topk} : " + ", ".join(
            f"{name(l)} ({p*100:.1f}%)" for l, p in r["topk"]))
        lines.append("")
    ax[1].text(0.0, 1.0, "\n".join(lines), va="top", ha="left",
               family="monospace", fontsize=11, transform=ax[1].transAxes)
    plt.tight_layout(); plt.show()

    # Also print a compact text comparison.
    print(f"Image #{idx} | true = {name(true_label)}")
    print(f"  Teacher : pred={name(t['pred']):<28} correct={t['correct']!s:<5} "
          f"conf={t['conf']*100:5.1f}%  loss={t['loss']:.4f}")
    print(f"  Student : pred={name(s['pred']):<28} correct={s['correct']!s:<5} "
          f"conf={s['conf']*100:5.1f}%  loss={s['loss']:.4f}")
    return idx

In [ ]:
# Compare the frozen teacher vs. the best student (E5) on a random VALIDATION image.
# Re-run this cell to draw a different random bird; pass idx=<n> to fix the image.
_ = compare_on_single_image(student_e5, teacher, val_ds)

## 15. Hyperparameter Tuning

KD is sensitive to its hyperparameters. Sensible search ranges:

| Hyperparameter | Candidate values | Effect |
|----------------|------------------|--------|
| **Temperature $T$** | 2, 4, 6 | Higher → softer targets, more dark knowledge. |
| **$\alpha$ / $\beta$** | 0.7/0.3, 0.5/0.5, 0.3/0.7 | Trade-off hard labels vs. teacher mimicry. |
| **$\gamma$ (AT)** | 10, 50, 100, 200 | Strength of attention matching. |
| **Learning rate** | 1e-5, 5e-5, 1e-4, 3e-4 | Optimisation speed/stability. |
| **Batch size** | 16, 32, 64 | Gradient noise / memory. |
| **Weight decay** | 1e-5, 1e-4, 5e-4 | Regularisation strength. |

> **🔑 Golden rule:** select hyperparameters using **validation accuracy only**.
> The **test set must never** influence any tuning decision — touch it once, at
> the very end, to report final numbers. Otherwise results are optimistically
> biased and not scientifically valid.

Example sketch of a validation-only sweep (run with reduced epochs to save time):

```python
for T in [2, 4, 6]:
    CFG.temperature = T
    h = train_kd(teacher, f"ckpt_tmp_T{T}.pt", tag=f"sweep-T{T}",
                 use_kd=True, use_at=True, num_epochs=10)
    print("T", T, "best val top1:", max(h["val_top1"]))
# pick the T with the highest *validation* top1, then retrain fully.
```

## 16. Conclusion

Using the results table above, summarise the findings:

* **Did KD improve MobileNetV2 over the baseline (E2)?**
  Compare E3/E4/E5 top-1 against E2. Distillation typically lifts the student by
  a few points at **zero extra inference cost** — the student stays the same size
  and speed.
* **Did attention transfer help fine-grained classification (E4)?**
  Because CUB classes differ in small parts, guiding the student to the teacher's
  focus regions usually helps it pick up discriminative cues.
* **Did the combined method (E5) perform best?**
  Response KD (*what* to predict) and attention transfer (*where* to look) are
  complementary, so E5 is generally the strongest student.

### Limitations
* **Training cost** — KD requires a fully trained teacher plus extra teacher
  forward passes each step.
* **Dependency on a strong teacher** — the student can only inherit knowledge the
  teacher actually has; a weak teacher caps student gains.
* **Sensitivity to KD hyperparameters** — $T$, $\alpha/\beta$ and especially
  $\gamma$ noticeably affect results and need validation-based tuning.

### Future work
* **FitNets** — match intermediate feature *representations* via a regressor.
* **RKD (Relational KD)** — preserve distances/angles between samples in feature
  space *(intentionally left out of this implementation; promising extension)*.
* **CRD (Contrastive Representation Distillation)** — contrastive objective for
  richer transfer.
* **Multi-layer attention transfer** — match attention at several depths, not
  just the final block.
* **Bounding-box-guided training** — exploit CUB's part/box annotations to focus
  learning on the bird region.

---
*Notebook complete — a full KD pipeline from data preparation to final
comparison, suitable for a university project or thesis.*